**Part 6 · Exercises** · step 24 of 25  
[← 22 Project: dynamic tool agent (part 2)](../05_Projects/22_dynamic_tool_agent/02_dynamic_tool_agent.ipynb) · [🏠 Roadmap](../README.md) · [24 Capstone project →](24_capstone_expense_assistant/README.md)

# 23 · Concept Drills — revise everything in one sitting

15 small drills, one per key concept, in course order, plus a "when to use what" quiz.

**How to use**
1. Run the setup cell.
2. For each drill: read the task → fill in the `TODO` cell → run the **check** cell.
   ✅ means you got it; an `AssertionError` tells you what's wrong.
3. Stuck? Open the 💡 solution, then re-type it yourself instead of copy-pasting.

Everything runs **offline** with fake models (`FakeListChatModel`, `FakeToolModel`), so it's free to repeat.
The scripted fake replies are a habit worth keeping: it's how you unit-test LLM apps (the capstone tests use it too).

In [1]:
# Run this first. Fake models let every drill run OFFLINE — no API key, no cost.
import operator
from typing import Annotated, Literal

from langchain_core.language_models.fake_chat_models import FakeListChatModel, FakeMessagesListChatModel
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from pydantic import ValidationError
from typing_extensions import TypedDict


class FakeToolModel(FakeMessagesListChatModel):
    """Replays scripted AIMessages in order and accepts bind_tools, so agents run offline."""

    def bind_tools(self, tools, **kwargs):
        return self


def tool_call(name: str, call_id: str = "call_1", **args) -> AIMessage:
    """What a real model returns when it wants to run a tool."""
    return AIMessage("", tool_calls=[{"name": name, "args": args, "id": call_id}])


print("Setup done ✅")

Setup done ✅


## Part 1 · Fundamentals

### D1 · Pydantic validation
*Revise: step 03*

Create a `Product` model: `name` (str, at least 2 chars), `price` (float, must be > 0), `tags` (list of str, default empty list).

In [17]:
from pydantic import BaseModel, Field
from typing import Optional

class Product(BaseModel):
    """TODO: name, price, tags"""
    name: str = Field(min_length=2)
    price: float = Field(gt=0)
    tags: Optional[list[str]] = []

In [20]:
p = Product(name="Pen", price=10)
assert p.tags == []
for bad in [dict(name="P", price=10), dict(name="Pen", price=0)]:
    try:
        Product(**bad)
        raise AssertionError(f"{bad} should be rejected")
    except ValidationError:
        pass
print("D1 ✅")

D1 ✅


<details><summary>💡 Solution (try first!)</summary>

```python
from pydantic import BaseModel, Field


class Product(BaseModel):
    """A product with validated name and price."""

    name: str = Field(min_length=2)
    price: float = Field(gt=0)
    tags: list[str] = []
```
</details>

### D2 · Literal & a router function
*Revise: step 03, 16*

Write `next_light(color)` for a traffic light: red→green, green→yellow, yellow→red. Annotate the parameter and return type with `Literal`, the same pattern LangGraph routers use.

In [28]:
def next_light(color)-> Literal["red","green","yellow"]:
    """Return the next traffic-light colour."""
    # print(color)
    if color=="red":
        return "green"
    elif color == "green":
        return "yellow"
    else:
        return "red"

In [29]:
assert [next_light(c) for c in ("red", "green", "yellow")] == ["green", "yellow", "red"]
assert "Literal" in str(next_light.__annotations__.get("return")), "annotate the return type with Literal"
print("D2 ✅")

D2 ✅


<details><summary>💡 Solution (try first!)</summary>

```python
Color = Literal["red", "green", "yellow"]


def next_light(color: Color) -> Color:
    """Return the next traffic-light colour."""
    return {"red": "green", "green": "yellow", "yellow": "red"}[color]
```
</details>

## Part 2 · LangChain

### D3 · Messages
*Revise: step 05*

Build `conversation`: a system message *You are a math tutor*, the user asks *What is 2+2?*, the AI answered *4*, and the user now asks *And times 3?*.

In [30]:
from langchain.messages import  AIMessage,HumanMessage,SystemMessage

In [31]:
conversation = [
    SystemMessage("you are a math tutor"),
    HumanMessage("what is 2+2"),
    AIMessage("4"),
    HumanMessage("And times 3")
]  

In [32]:
assert [type(m) for m in conversation] == [SystemMessage, HumanMessage, AIMessage, HumanMessage]
assert conversation[2].content == "4"
print("D3 ✅")

D3 ✅


<details><summary>💡 Solution (try first!)</summary>

```python
conversation = [
    SystemMessage("You are a math tutor"),
    HumanMessage("What is 2+2?"),
    AIMessage("4"),
    HumanMessage("And times 3?"),
]
```
</details>

### D4 · ChatPromptTemplate + MessagesPlaceholder
*Revise: step 06*

Create `prompt` with: system *You are a {role}.*, a `history` placeholder, and human *{question}*.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = None  # TODO

In [ ]:
assert set(prompt.input_variables) == {"role", "history", "question"}
msgs = prompt.invoke({"role": "chef", "history": [HumanMessage("hi"), AIMessage("hello")], "question": "Pasta?"}).to_messages()
assert len(msgs) == 4 and msgs[0].content == "You are a chef." and msgs[-1].content == "Pasta?"
print("D4 ✅")

<details><summary>💡 Solution (try first!)</summary>

```python
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}."),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])
```
</details>

### D5 · An LCEL chain
*Revise: step 06*

Make `chain = prompt | model | parser` so that `chain.invoke({...})` returns a **plain string**. Use `FakeListChatModel(responses=["Paris"])` as the model.

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

capital_prompt = ChatPromptTemplate.from_messages([("human", "Capital of {country}?")])
chain = None  # TODO

In [ ]:
assert chain.invoke({"country": "France"}) == "Paris"
print("D5 ✅")

<details><summary>💡 Solution (try first!)</summary>

```python
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

capital_prompt = ChatPromptTemplate.from_messages([("human", "Capital of {country}?")])
chain = capital_prompt | FakeListChatModel(responses=["Paris"]) | StrOutputParser()
```
</details>

### D6 · Conversation history by hand
*Revise: step 08*

Complete `chat(history, text, model)`: append the user's message, call the model with the **whole** history, append the reply, and return its text.

In [ ]:
def chat(history: list, text: str, model) -> str:
    """Send one turn; mutate ``history`` in place."""
    ...  # TODO

In [ ]:
model = FakeListChatModel(responses=["Hi Ujjawal!", "Your name is Ujjawal."])
history = [SystemMessage("Be brief.")]
assert chat(history, "I'm Ujjawal", model) == "Hi Ujjawal!"
assert chat(history, "My name?", model) == "Your name is Ujjawal."
assert [type(m).__name__ for m in history] == ["SystemMessage", "HumanMessage", "AIMessage", "HumanMessage", "AIMessage"]
print("D6 ✅")

<details><summary>💡 Solution (try first!)</summary>

```python
def chat(history: list, text: str, model) -> str:
    """Send one turn; mutate ``history`` in place."""
    history.append(HumanMessage(text))
    reply = model.invoke(history)
    history.append(reply)
    return reply.content
```
</details>

### D7 · Create a tool
*Revise: step 09*

Create a tool `convert_currency(amount: float, rate: float) -> float` that multiplies the two numbers. Give it a docstring (the LLM reads it!).

In [ ]:
from langchain.tools import tool

# TODO: @tool ...

In [ ]:
assert convert_currency.name == "convert_currency"
assert set(convert_currency.args) == {"amount", "rate"}
assert convert_currency.description, "add a docstring"
assert convert_currency.invoke({"amount": 10, "rate": 83}) == 830
print("D7 ✅")

<details><summary>💡 Solution (try first!)</summary>

```python
from langchain.tools import tool


@tool
def convert_currency(amount: float, rate: float) -> float:
    """Convert an amount of money using an exchange rate."""
    return amount * rate
```
</details>

### D8 · The tool loop by hand
*Revise: step 09*

The model replied with `ai_msg` (a tool call). Execute every tool call and collect the resulting `ToolMessage`s in `tool_messages`. Hint: `convert_currency.invoke(tool_call)` returns a ToolMessage.

In [ ]:
ai_msg = tool_call("convert_currency", call_id="abc", amount=10, rate=83)
tool_messages = []  # TODO

In [ ]:
assert len(tool_messages) == 1 and isinstance(tool_messages[0], ToolMessage)
assert tool_messages[0].tool_call_id == "abc" and tool_messages[0].content == "830.0"
print("D8 ✅")

<details><summary>💡 Solution (try first!)</summary>

```python
ai_msg = tool_call("convert_currency", call_id="abc", amount=10, rate=83)
tool_messages = [convert_currency.invoke(call) for call in ai_msg.tool_calls]
```
</details>

### D9 · create_agent
*Revise: step 10*

Build `agent` with `create_agent`, using `scripted_model` and the `convert_currency` tool, then invoke it with the question. Store the result in `result`.

In [ ]:
from langchain.agents import create_agent

scripted_model = FakeToolModel(responses=[
    tool_call("convert_currency", amount=10, rate=83),  # 1st model call: "please run the tool"
    AIMessage("10 USD is 830 INR."),                     # 2nd model call: final answer
])
question = {"messages": [HumanMessage("Convert 10 USD to INR at 83")]}
agent = None   # TODO
result = None  # TODO

In [ ]:
assert [type(m).__name__ for m in result["messages"]] == ["HumanMessage", "AIMessage", "ToolMessage", "AIMessage"]
assert result["messages"][-1].content == "10 USD is 830 INR."
print("D9 ✅  the agent ran the loop: model → tool → model")

<details><summary>💡 Solution (try first!)</summary>

```python
from langchain.agents import create_agent

scripted_model = FakeToolModel(responses=[
    tool_call("convert_currency", amount=10, rate=83),
    AIMessage("10 USD is 830 INR."),
])
question = {"messages": [HumanMessage("Convert 10 USD to INR at 83")]}
agent = create_agent(scripted_model, tools=[convert_currency])
result = agent.invoke(question)
```
</details>

### D10 · Memory with a checkpointer
*Revise: step 11*

Give `memory_agent` memory. Send two messages in thread `"a"` and one in thread `"b"`.

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# one distinct reply per turn (a reused message keeps its id, so add_messages would REPLACE it)
model = FakeToolModel(responses=[AIMessage("hi!"), AIMessage("hi again!"), AIMessage("hello!")])
memory_agent = None  # TODO: create_agent(..., checkpointer=...)

# TODO: 2 invokes with thread_id "a", 1 invoke with thread_id "b"

In [ ]:
count = lambda t: len(memory_agent.get_state({"configurable": {"thread_id": t}}).values["messages"])
assert (count("a"), count("b")) == (4, 2), "thread a: 2 turns x 2 messages; thread b: 1 turn"
print("D10 ✅")

<details><summary>💡 Solution (try first!)</summary>

```python
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# one distinct reply per turn (a reused message keeps its id, so add_messages would REPLACE it)
model = FakeToolModel(responses=[AIMessage("hi!"), AIMessage("hi again!"), AIMessage("hello!")])
memory_agent = create_agent(model, checkpointer=InMemorySaver())

for thread, text in [("a", "hi"), ("a", "again"), ("b", "hello")]:
    memory_agent.invoke({"messages": [HumanMessage(text)]}, {"configurable": {"thread_id": thread}})
```
</details>

### D11 · Human-in-the-loop
*Revise: step 14*

Make the agent pause before `convert_currency`, then **reject** the call. Store the first result in `paused` and the result after resuming in `final`.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

model = FakeToolModel(responses=[tool_call("convert_currency", amount=1, rate=83), AIMessage("Okay, cancelled.")])
config = {"configurable": {"thread_id": "hitl"}}
hitl_agent = None  # TODO (remember: HITL needs a checkpointer)
paused = None      # TODO: invoke with a question
final = None       # TODO: resume with a reject decision

In [ ]:
assert "__interrupt__" in paused
tool_results = [m.content for m in final["messages"] if isinstance(m, ToolMessage)]
assert "83.0" not in tool_results, "the tool must not run after a reject"
print("D11 ✅")

<details><summary>💡 Solution (try first!)</summary>

```python
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

model = FakeToolModel(responses=[tool_call("convert_currency", amount=1, rate=83), AIMessage("Okay, cancelled.")])
config = {"configurable": {"thread_id": "hitl"}}
hitl_agent = create_agent(
    model,
    tools=[convert_currency],
    checkpointer=InMemorySaver(),
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"convert_currency": True})],
)
paused = hitl_agent.invoke({"messages": [HumanMessage("convert 1 USD")]}, config)
final = hitl_agent.invoke(Command(resume={"decisions": [{"type": "reject", "message": "not now"}]}), config)
```
</details>

### D12 · Custom middleware
*Revise: step 15*

Write `CountingMiddleware`, which counts model calls in `self.count` using a `before_model` hook.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware


class CountingMiddleware(AgentMiddleware):
    """TODO"""

In [ ]:
counter = CountingMiddleware()
model = FakeToolModel(responses=[tool_call("convert_currency", amount=2, rate=2), AIMessage("4")])
create_agent(model, tools=[convert_currency], middleware=[counter]).invoke({"messages": [HumanMessage("2*2")]})
assert counter.count == 2, "one model call before the tool, one after"
print("D12 ✅")

<details><summary>💡 Solution (try first!)</summary>

```python
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware


class CountingMiddleware(AgentMiddleware):
    """Count how many times the model is called."""

    def __init__(self):
        super().__init__()
        self.count = 0

    def before_model(self, state, runtime):
        self.count += 1
        return None
```
</details>

## Part 3 · LangGraph

### D13 · A graph with a conditional edge
*Revise: step 16*

Build `number_graph`: `START → check → (even | odd) → END`. Node `even` sets `label="even"`, node `odd` sets `label="odd"`.

In [ ]:
from langgraph.graph import END, START, StateGraph


class NumberState(TypedDict):
    n: int
    label: str


number_graph = None  # TODO: nodes, router, edges, compile()

In [ ]:
assert number_graph.invoke({"n": 4, "label": ""})["label"] == "even"
assert number_graph.invoke({"n": 7, "label": ""})["label"] == "odd"
print("D13 ✅")

<details><summary>💡 Solution (try first!)</summary>

```python
from langgraph.graph import END, START, StateGraph


class NumberState(TypedDict):
    n: int
    label: str


def route(state: NumberState) -> Literal["even", "odd"]:
    """Pick the next node from the number's parity."""
    return "even" if state["n"] % 2 == 0 else "odd"


builder = StateGraph(NumberState)
builder.add_node("check", lambda state: {})  # does nothing, routing happens on its edge
builder.add_node("even", lambda state: {"label": "even"})
builder.add_node("odd", lambda state: {"label": "odd"})
builder.add_edge(START, "check")
builder.add_conditional_edges("check", route)
builder.add_edge("even", END)
builder.add_edge("odd", END)
number_graph = builder.compile()
```
</details>

### D14 · Reducers
*Revise: step 17*

Fix `LogState` so that `log` **accumulates** instead of being overwritten. Only change the type hint.

In [ ]:
from langgraph.graph import END, START, StateGraph


class LogState(TypedDict):
    log: list[str]  # TODO: add a reducer


b = StateGraph(LogState)
b.add_node("a", lambda s: {"log": ["a"]})
b.add_node("b", lambda s: {"log": ["b"]})
b.add_edge(START, "a"); b.add_edge("a", "b"); b.add_edge("b", END)
log_graph = b.compile()

In [ ]:
assert log_graph.invoke({"log": ["start"]})["log"] == ["start", "a", "b"]
print("D14 ✅")

<details><summary>💡 Solution (try first!)</summary>

```python
from langgraph.graph import END, START, StateGraph


class LogState(TypedDict):
    log: Annotated[list[str], operator.add]


b = StateGraph(LogState)
b.add_node("a", lambda s: {"log": ["a"]})
b.add_node("b", lambda s: {"log": ["b"]})
b.add_edge(START, "a"); b.add_edge("a", "b"); b.add_edge("b", END)
log_graph = b.compile()
```
</details>

### D15 · ReAct loop as a graph
*Revise: step 18*

Build `react_graph` by hand: an `llm` node that calls `react_model`, a `tools` node (`ToolNode`), `tools_condition` after `llm`, and a loop back from `tools` to `llm`.

In [ ]:
from langgraph.graph import START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

react_model = FakeToolModel(responses=[tool_call("convert_currency", amount=5, rate=2), AIMessage("5 x 2 = 10")])
react_graph = None  # TODO

In [ ]:
out = react_graph.invoke({"messages": [HumanMessage("convert 5 at rate 2")]})
assert [type(m).__name__ for m in out["messages"]] == ["HumanMessage", "AIMessage", "ToolMessage", "AIMessage"]
assert out["messages"][2].content == "10.0"
print("D15 ✅  you rebuilt create_agent by hand!")

<details><summary>💡 Solution (try first!)</summary>

```python
from langgraph.graph import START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

react_model = FakeToolModel(responses=[tool_call("convert_currency", amount=5, rate=2), AIMessage("5 x 2 = 10")])


def llm(state: MessagesState) -> dict:
    """Call the model; return only the new message."""
    return {"messages": [react_model.invoke(state["messages"])]}


g = StateGraph(MessagesState)
g.add_node("llm", llm)
g.add_node("tools", ToolNode([convert_currency]))
g.add_edge(START, "llm")
g.add_conditional_edges("llm", tools_condition)
g.add_edge("tools", "llm")
react_graph = g.compile()
```
</details>

## Part 4 · When to use what — quick quiz (step 20)
Answer in your head first, then open the answer.

1. You need to summarise 500 support tickets, the same way every time.
   <details><summary>Answer</summary>A <b>chain</b> (<code>prompt | model | parser</code>) with <code>.batch()</code>. Fixed steps, no tools, so no agent.</details>

2. Users ask free-form questions about orders stored in Postgres.
   <details><summary>Answer</summary><code>create_agent</code> + a read-only SQL tool (like project 21). The model must decide which queries to run.</details>

3. A support bot must classify the request, then either refund (needs a manager's approval), escalate, or answer FAQs.
   <details><summary>Answer</summary><b>LangGraph</b>: a router node with conditional edges, with an interrupt/HITL before the refund step. That's custom control flow.</details>

4. Chats run for hours and get expensive.
   <details><summary>Answer</summary><code>SummarizationMiddleware</code> (or trimming), plus a cheaper model for the summary.</details>

5. The agent must never see customers' card numbers.
   <details><summary>Answer</summary><code>PIIMiddleware("credit_card", strategy="mask" | "redact" | "block")</code>.</details>

6. Why does `"what is my name?"` fail on a fresh `create_agent` without a checkpointer?
   <details><summary>Answer</summary>LLMs are stateless and nothing saved the earlier turn. Add <code>checkpointer=InMemorySaver()</code> and reuse the same <code>thread_id</code>.</details>

7. When is the raw OpenAI SDK fine?
   <details><summary>Answer</summary>A tiny script tied to one provider, or a brand-new provider feature. Otherwise LangChain saves boilerplate and lets you switch providers.</details>

## Done?
If every check printed ✅, you know the building blocks. Next, put them **together** in a real project:
the capstone (step 24).

---
✅ **Done with 23 Concept drills.** Next: [24 Capstone project →](24_capstone_expense_assistant/README.md)  
[🏠 Roadmap](../README.md)